# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## **This notebook is a specification of data contract for a time-aware page-priority system.**

**Works:** pages are observed every day; constructs a recent state for each page; estimates the decline/risk from historic behavior; outputs a ranked list of pages to pay attention to.


## 1. Unit of analysis + time window

**Raw data grain:** One row represents ***report_date × client_hash_id × content_hash_id.***

**Modeling/scoring grain:** One row represents client_hash_id × content_hash_id × scoring_date.

A page can therefore appear repeatedly over time. The same page is not represented by one permanent training row.

**Time principle:** Features for a scoring date may use only observations available on or before that scoring date. Future observations are reserved for label construction and evaluation.

**Time window:** The initial feature window is 90 days. This is a feature-window choice; older warehouse history is not discarded



In [5]:
import os
from getpass import getpass
import duckdb
import pandas as pd
import numpy as np

# Get Hugging Face token
hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    hf_token = getpass("Enter your Hugging Face READ token: ")

# Connect to DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Pass token to DuckDB
con.execute(
    "SET VARIABLE hf_token = ?",
    [hf_token]
)

# Create Hugging Face secret
con.execute("""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# Dataset paths
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

MONTHS = [
    "2026-01",
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06"
]

HISTORY_DAYS = 90

print("Warehouse connection ready.")
print("Initial feature window:", HISTORY_DAYS, "days")

Enter your Hugging Face READ token: ··········
Warehouse connection ready.
Initial feature window: 90 days


## 2. Fields: feature / label / context / excluded
## ***Make use of the information provided by the scoring date:***
- impressions
- clicks
Click rate or CTR that was calculated from the number of clicks and impressions.
- impression-weighted position
- recent change/trend
- volatility
- persistence/frequency of decline

## Label
The target is based on an observed future behavior. Must NOT be copied from a downstream decision field.

The actual future assessment period and the proxy threshold is a validation decision.

## Context
- page/client identifiers
- scoring date
- data-availability indicators
- number of observed days in the feature window

## Excluded
- future-period metrics
- target/label columns
- downstream priority/health/action fields
- raw URLs, queries, keywords, client names
Only identifiers that are used in a JOIN or GROUP BY are allowed.
Such exclusions guard against target and decision leakage.

In [7]:
available = set(schema["column_name"].astype(str))

feature_fields = [
    c for c in [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position"
    ]
    if c in available
]

context_fields = [
    c for c in [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_data_available",
        "ga4_data_available"
    ]
    if c in available
]

excluded_fields = [
    c for c in [
        "trend_direction",
        "trend_pct",
        "health_score",
        "priority_score",
        "action_type",
        "refresh_tier",
        "recommended_action"
    ]
    if c in available
]

rows = []

for c in feature_fields:
    rows.append({
        "field": c,
        "category": "historical_feature",
        "reason": "Available by the scoring date."
    })

for c in context_fields:
    rows.append({
        "field": c,
        "category": "context",
        "reason": "Used for grouping, ordering, availability, or validation."
    })

rows.append({
    "field": "target",
    "category": "label",
    "reason": "Created from a later observed outcome during training."
})

for c in excluded_fields:
    rows.append({
        "field": c,
        "category": "excluded",
        "reason": "Downstream outcome/decision or outside the page-level feature contract."
    })

field_contract = pd.DataFrame(rows).drop_duplicates(subset="field")

display(field_contract)

,field,category,reason
0,gsc_impressions,historical_feature,Available by the scoring date.
1,gsc_clicks,historical_feature,Available by the scoring date.
2,gsc_sum_position,historical_feature,Available by the scoring date.
3,report_date,context,"Used for grouping, ordering, availability, or ..."
4,client_hash_id,context,"Used for grouping, ordering, availability, or ..."
5,content_hash_id,context,"Used for grouping, ordering, availability, or ..."
6,gsc_data_available,context,"Used for grouping, ordering, availability, or ..."
7,ga4_data_available,context,"Used for grouping, ordering, availability, or ..."
8,target,label,Created from a later observed outcome during t...


## 3. Verify it with queries (grain, counts, missing values, windows)
To ensure the following is checked:
- the declared daily grain
- duplicate grain keys
- available date windows
- page/client counts
- GSC availability
- When there is no value for a primary GSC field, it appears as a missing value.

In [11]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        (
            SELECT COUNT(*)
            FROM (
                SELECT
                    report_date,
                    client_hash_id,
                    content_hash_id
                FROM read_parquet('{FACT}/month=2026-06/*.parquet')
                GROUP BY
                    report_date,
                    client_hash_id,
                    content_hash_id
            )
        ) AS distinct_grain_keys,
        COUNT(*) -
        (
            SELECT COUNT(*)
            FROM (
                SELECT
                    report_date,
                    client_hash_id,
                    content_hash_id
                FROM read_parquet('{FACT}/month=2026-06/*.parquet')
                GROUP BY
                    report_date,
                    client_hash_id,
                    content_hash_id
            )
        ) AS duplicate_grain_rows
    FROM read_parquet('{FACT}/month=2026-06/*.parquet')
""").df()

display(grain_check)

duplicate_keys = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{FACT}/month=2026-06/*.parquet')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 20
""").df()

print("Duplicated grain keys:")
display(duplicate_keys)

,rows,distinct_grain_keys,duplicate_grain_rows
0,11694072,11687682,6390


Duplicated grain keys:


,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-13,client_e00b29e582949543,content_e3491394a9f3e2b3,2
1,2026-06-13,client_e00b29e582949543,content_83b20e2cd4d5437f,2
2,2026-06-13,client_e00b29e582949543,content_04307a138472fc03,2
3,2026-06-13,client_e00b29e582949543,content_638ac5ad8023d177,2
4,2026-06-13,client_e00b29e582949543,content_ff881e76e85001f4,2
5,2026-06-13,client_810019792c9b8efc,content_097eada60a3a3f2b,2
6,2026-06-13,client_810019792c9b8efc,content_fa60e9dea96f683a,2
7,2026-06-13,client_810019792c9b8efc,content_dacee0e0bac2dd31,2
8,2026-06-13,client_810019792c9b8efc,content_034d4ed04b3ddd55,2
9,2026-06-13,client_810019792c9b8efc,content_41a28b3332556857,2


## 4. Data limits

- The warehouse will be longitudinal, meaning pages may have varying numbers of available histories.

- An observation that is missing is not necessarily considered zero activity, and should not be assumed as such.

- The feature window has been recent, 90 days, so that many of the same observations will be in both windows.

- Less useful observations may be found in the early history of a page.

- Data can be used to describe the performance and direction of the market in the past but can't predict what will happen in the future until it does happen.

- The feature state is the same as the recently defined feature but older history is stored in the warehouse.


In [13]:
parts = []

for month in MONTHS:
    parts.append(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            DATE_TRUNC('month', report_date) AS month,
            SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS measured_days
        FROM read_parquet('{FACT}/month={month}/*.parquet')
        GROUP BY client_hash_id, content_hash_id, DATE_TRUNC('month', report_date)
    """)

page_month_history = con.sql(
    " UNION ALL ".join(parts)
).df()

print("Page-month history:")
print(f"Total page-month records: {len(page_month_history):,}")
print(
    f"Minimum measured days: "
    f"{page_month_history['measured_days'].min()}"
)
print(
    f"Maximum measured days: "
    f"{page_month_history['measured_days'].max()}"
)
print(
    f"Average measured days: "
    f"{page_month_history['measured_days'].mean():.2f}"
)

display(
    page_month_history[
        ["measured_days"]
    ].describe()
)

Page-month history:
Total page-month records: 2,075,497
Minimum measured days: 0.0
Maximum measured days: 31.0
Average measured days: 10.01


,measured_days
count,2.075497e+06
mean,1.001370e+01
std,1.266467e+01
min,0.000000e+00
25%,0.000000e+00
50%,1.000000e+00
75%,2.500000e+01
max,3.100000e+01


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.